# Kata 18 — Scratchpad Persistente

> Spec: [`specs/018-scratchpad-persistence/spec.md`](../../specs/018-scratchpad-persistence/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap
import json, pathlib

client, settings = bootstrap(budget_calls=8)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Memoria conversacional es volátil: `/compact` o reset la borran. El **scratchpad** es un archivo de markdown estructurado donde el agente vuelca conclusiones validadas. Sobrevive a compactación y a sesiones nuevas.

## 2. Por qué importa

Un descubrimiento valioso del turno 30 desaparece si la sesión se compacta y vivía sólo en la conversación. El scratchpad es la única memoria persistente que el agente controla.

## 3. Ejemplo correcto — tool de escritura + lectura entre sesiones

In [ ]:
SCRATCHPAD = pathlib.Path("./investigation-scratchpad.md")
SCRATCHPAD.write_text("# Scratchpad\n\n## Decisiones\n\n## Hallazgos\n\n## Pendientes\n")

SCRATCH_TOOL = {
    "name": "append_scratchpad",
    "description": "Anexa una entrada validada al scratchpad bajo una sección.",
    "input_schema": {
        "type": "object",
        "properties": {
            "section": {"type": "string", "enum": ["Decisiones","Hallazgos","Pendientes"]},
            "entry": {"type": "string", "minLength": 5},
        },
        "required": ["section","entry"],
    },
}

def append_scratchpad(args):
    text = SCRATCHPAD.read_text()
    target = f"## {args['section']}"
    parts = text.split(target, 1)
    if len(parts) != 2:
        return {"error": "section_not_found"}
    body = parts[1]
    next_section_idx = body.find("\n## ")
    insert_at = next_section_idx if next_section_idx >= 0 else len(body)
    new_body = body[:insert_at].rstrip() + f"\n- {args['entry']}\n" + body[insert_at:]
    SCRATCHPAD.write_text(parts[0] + target + new_body)
    return {"ok": True}

system = (
    "Eres un agente investigando un repo. Tienes la herramienta append_scratchpad. "
    "Cuando confirmes una decisión o hallazgo, llámala. Mantén entradas concisas."
)
messages = [{"role":"user","content":"Decidí usar pydantic v2. Encontré un bug en src/parser.py línea 142."}]

resp = client.messages.create(system=system, messages=messages, tools=[SCRATCH_TOOL])
for b in resp.content:
    if b.type == "tool_use":
        print(b.name, b.input)
        print("→", append_scratchpad(b.input))

print("\n=== contenido scratchpad ===")
print(SCRATCHPAD.read_text())

### 3.1 Sesión nueva: el scratchpad es el contexto

In [ ]:
def new_session_with_scratchpad(question: str):
    scratch_text = SCRATCHPAD.read_text()
    sys = (
        "Eres un agente que continúa una investigación. "
        "Este es tu scratchpad persistente con decisiones y hallazgos previos:\n\n"
        f"{scratch_text}\n\n"
        "Apóyate en él. Si descubres algo nuevo, usa append_scratchpad."
    )
    return client.messages.create(system=sys, messages=[{"role":"user","content": question}], tools=[SCRATCH_TOOL])

resp2 = new_session_with_scratchpad("¿Qué decidimos sobre la versión de pydantic?")
print(next((b.text for b in resp2.content if b.type == "text"), "")[:300])

La nueva sesión nunca conoció el turno original; el scratchpad le dio el contexto.

## 4. Anti-patrón — todo en la conversación

In [ ]:
conv = [
    {"role":"user","content":"Decidí usar pydantic v2."},
    {"role":"assistant","content":"OK, lo registro."},
    # ... 50 turnos más ...
]
# Si el cliente compacta agresivamente o reinicia, "pydantic v2" se pierde
print("turnos en conversación:", len(conv))
print("turnos tras compactación agresiva (drop intermedios):", len(conv[:1] + conv[-1:]))
# La decisión sobre pydantic estaba en el turno 0; si lo compactan, desaparece

## 5. Argumento de certificación

- **Memoria volátil vs persistente**: la conversación es la primera; el scratchpad la segunda.
- **Estructurado por secciones** (Decisiones / Hallazgos / Pendientes): el agente sabe dónde escribir y leer.
- **Entradas concisas y validadas**: nunca monólogo, sólo conclusiones que aporten en sesiones futuras.
- **Conexión con otros katas**: complementa la compactación de Kata 11; en investigación adaptativa (Kata 19) el scratchpad guarda los hallazgos del coordinador.

## 6. Auto-evaluación

**1. ¿Qué pasa si el scratchpad y la conversación se contradicen?**

Gana el scratchpad. La conversación es contexto reciente y volátil; el scratchpad es memoria curada. Si la contradicción es genuina, el agente debe actualizar el scratchpad con una nueva entrada que reemplaza la anterior.

**2. ¿Cuándo el agente debe **borrar** entradas del scratchpad?**

Cuando un hallazgo se invalida (ej. un bug reportado se confirma como falso positivo). En la práctica, agrego una entrada que invalida la anterior antes que borrar — preservar el rastro de la decisión.

**3. ¿Cómo verifico que un hallazgo sobrevivió a `/compact`?**

Inicio una sesión nueva (sin pasar `messages` previa), incluyo el scratchpad en el system prompt, y le pregunto al modelo por el hallazgo. Si lo recuerda, sobrevivió. La celda §3.1 hace exactamente eso.